# Pregunta 1: orquestador de Colab (pasos 03, 04 y 05)

Ejecuta en orden los 6 entrenamientos, los 2 Grad-CAM y el análisis final. Todo se guarda en `ejecuciones/<ID_EJECUCION>/`.

**Antes de empezar**
1. Subir a Google Drive la carpeta `Pregunta01` completa, con `datos/` y `tensores/` generados en la parte local.
2. Activar la GPU: *Entorno de ejecución > Cambiar tipo de entorno de ejecución > GPU (T4) > Guardar*.
3. Ajustar `RUTA_PROYECTO` e `ID_EJECUCION` en la celda 1 y ejecutar todo: *Entorno de ejecución > Ejecutar todas*.

## 1. Parámetros

In [5]:
# Carpeta Pregunta01 dentro de tu Google Drive
RUTA_PROYECTO = "/content/drive/MyDrive/Semestre 2/Redes Neuronales/Examen Parcial/Desarrollo/Pregunta 01"

# Nombre de la corrida: cada corrida se guarda en ejecuciones/<ID_EJECUCION>/ y ninguna pisa a otra
ID_EJECUCION = "corrida_01"

# Para retomar una corrida que falló, poner aquí el paso desde el que se continúa (por ejemplo "gradcam_base").
# None = corrida completa desde el inicio.
DESDE = None

## 2. Conexión con Drive y verificación de requisitos

In [6]:
import os, sys, shutil, subprocess, time
from datetime import datetime
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive", force_remount=True)
os.chdir(RUTA_PROYECTO)
sys.path.insert(0, RUTA_PROYECTO)
os.environ["ID_EJECUCION"] = ID_EJECUCION          # los pasos 03, 04 y 05 leen la corrida de aquí
os.environ["PYTHONIOENCODING"] = "utf-8"
import configuracion as C

DIR_EJECUCION = Path("ejecuciones") / ID_EJECUCION
problemas = []
if not C.CSV_SELECCION.exists():
    problemas.append(f"Falta {C.CSV_SELECCION}: sube la carpeta datos/ generada en la parte local.")
if not any((C.DIR_TENSORES / "principal").glob("*.npz")):
    problemas.append("No hay volúmenes en tensores/principal: sube la carpeta tensores/ de la parte local.")
if DESDE is None and (DIR_EJECUCION / "experimentos").exists():
    problemas.append(f"{DIR_EJECUCION} ya tiene experimentos: cambia ID_EJECUCION o usa DESDE para continuar.")
if shutil.which("nvidia-smi") is None:
    problemas.append("No hay GPU: Entorno de ejecución > Cambiar tipo de entorno > GPU (T4).")
if problemas:
    raise SystemExit("No se puede empezar:\n  - " + "\n  - ".join(problemas))

# dependencias que Colab no trae
for paquete in ["pydicom", "timm"]:
    if subprocess.run([sys.executable, "-c", f"import {paquete}"], capture_output=True).returncode:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquete], check=True)
print(f"Pipeline {C.VERSION} | corrida: {ID_EJECUCION} | carpeta: {os.getcwd()}")
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout)

Mounted at /content/drive
Pipeline v4 | corrida: corrida_01 | carpeta: /content/drive/MyDrive/Semestre 2/Redes Neuronales/Examen Parcial/Desarrollo/Pregunta 01
NVIDIA L4, 23034 MiB



## 3. Pasos de la corrida

In [ ]:
BASE = "resnet50_bce_suave_ninguno_principal"
CERO = "cero_bce_suave_ninguno_principal"

# orden fijo: clave -> (descripción, script, argumentos)
PASOS = {
    "cero":         ("CNN 2.5D desde cero",                      "03_entrenar.py", ["--arquitectura", "cero"]),
    "base":         ("ResNet-50 preentrenada (modelo base)",     "03_entrenar.py", ["--arquitectura", "resnet50"]),
    "focal":        ("Ablación 1: pérdida focal",                "03_entrenar.py", ["--arquitectura", "resnet50", "--perdida", "focal"]),
    "consenso":     ("Ablación 2: solo consenso",                "03_entrenar.py", ["--arquitectura", "resnet50", "--etiquetas", "consenso"]),
    "pos_weight":   ("Ablación 3: pos_weight",                   "03_entrenar.py", ["--arquitectura", "resnet50", "--balanceo", "pos_weight"]),
    "coronal":      ("Ablación 4: plano coronal",                "03_entrenar.py", ["--arquitectura", "resnet50", "--plano", "alterna"]),
    "gradcam_base": ("Grad-CAM del modelo base",                 "04_gradcam.py",  ["--experimento", BASE, "--arquitectura", "resnet50"]),
    "gradcam_cero": ("Grad-CAM de la CNN desde cero",            "04_gradcam.py",  ["--experimento", CERO, "--arquitectura", "cero"]),
    "analisis":     ("Intervalos de confianza, tabla y figuras", "05_analisis_informe.py", []),
}
claves = list(PASOS)[list(PASOS).index(DESDE):] if DESDE else list(PASOS)
if "coronal" in claves and not (C.DIR_TENSORES / "alterna").exists():
    print("Aviso: no existe tensores/alterna; se omite la ablación de plano coronal.")
    claves.remove("coronal")
print("Pasos:", ", ".join(claves))

def ejecutar_paso(clave, log):
    """Corre un paso en un proceso aparte, muestra y guarda su salida. Devuelve (código, segundos)."""
    descripcion, script, args = PASOS[clave]
    cabecera = f"\n{'=' * 70}\n[{clave}] {descripcion}\n$ python colab/{script} {' '.join(args)}\n{'=' * 70}"
    print(cabecera); log.write(cabecera + "\n")
    inicio = time.time()
    with subprocess.Popen([sys.executable, "-u", f"colab/{script}", *args], stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, encoding="utf-8", errors="replace") as proceso:
        for linea in proceso.stdout:
            print(linea, end=""); log.write(linea); log.flush()
    return proceso.returncode, time.time() - inicio

## 4. Ejecución
Mantener la pestaña activa: en una GPU T4 cada experimento tarda unos 6 minutos.

In [ ]:
dir_logs = DIR_EJECUCION / "logs"; dir_logs.mkdir(parents=True, exist_ok=True)
shutil.copy("configuracion.py", dir_logs / "configuracion_usada.py")
(dir_logs / "gpu.txt").write_text(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"],
                                                 capture_output=True, text=True).stdout)
resumen = []
with open(dir_logs / "log_colab.txt", "a", encoding="utf-8") as log:   # "a": al retomar se agrega al mismo log
    log.write(f"\n== Corrida {ID_EJECUCION} | inicio {datetime.now():%Y-%m-%d %H:%M:%S} | pasos: {', '.join(claves)}\n")
    for clave in claves:
        codigo, segundos = ejecutar_paso(clave, log)
        resumen.append((clave, "ok" if codigo == 0 else f"FALLÓ (código {codigo})", segundos))
        if codigo != 0:
            break
    tabla = "\n".join(f"  {c:<13} {e:<20} {s / 60:6.1f} min" for c, e, s in resumen)
    final = f"\n{'=' * 70}\nResumen de la corrida {ID_EJECUCION}\n{tabla}"
    print(final); log.write(final + "\n")

if resumen[-1][1] != "ok":
    raise SystemExit(f'Corrige el problema, pon DESDE = "{resumen[-1][0]}" en la celda 1 y vuelve a ejecutar todo.')

## 5. Descarga de resultados
Genera un zip con la corrida (sin los pesos `.pt`) y la parte generada de los datos.

In [ ]:
from google.colab import files
nombre_zip = f"resultados_{ID_EJECUCION}.zip"
subprocess.run(["zip", "-qr", nombre_zip, str(DIR_EJECUCION), "datos/generados", "-x", "*.pt"], check=True)
print(f"{nombre_zip} creado en {RUTA_PROYECTO}")
files.download(nombre_zip)